<a href="https://colab.research.google.com/github/amonap-trika/service-example-vtex-training/blob/main/Data_Table_Display.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import csv
import json
import requests
import os
from google.colab import files
from google.colab import drive


# -------- CONFIGURATION -------- #
VTEX_ACCOUNT = "helmethouseprod"         # e.g. helmet-house
VTEX_ENV = "myvtex.com"               # normally "myvtex.com"
DATA_ENTITY = "b2b_users"                    # Masterdata acronym (e.g., CL = Clients)
AUTH_TOKEN = "vtexappkey-helmethouseprod-YWQSXD:MJYLODJLUEEULZBJOHOIDMIBYBLODJZRLRHPGRYGNGLELDDGSOHUHOVAZKGBPHXMMATGKZCZRSODMTSQMKAKOMPNCAOOXYFSKFCKJWMHIWLHKLUAEVFCWMQDLHNHYPUO"  # Create from VTEX admin

INPUT_CSV = "/content/test_invalid_schema_users.csv"
OUTPUT_DIR = "output_json"
FIELDS_TO_UPDATE = {
    "canImpersonate":True
}
# -------------------------------- #

BASE_URL = f"https://{VTEX_ACCOUNT}.{VTEX_ENV}/api/dataentities/{DATA_ENTITY}/documents"

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "X-VTEX-API-AppKey": AUTH_TOKEN.split(":")[0],
    "X-VTEX-API-AppToken": AUTH_TOKEN.split(":")[1]
}



In [17]:
def safe_json(response):
    try:
        if response.text.strip():  # not empty
            return response.json()
        else:
            return {"message": "No content"}
    except Exception:
        return {"message": "Non-JSON response", "raw": response.text}


def fetch_document(user_id):
    url = f"{BASE_URL}/{user_id}?_fields=_all"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data = safe_json(response) # Use safe_json here
    print(data)
    return data


def update_document(user_id, fields):
    url = f"{BASE_URL}/{user_id}?_schema=v0.1.2"
    print(url)
    response = requests.patch(url, headers=HEADERS, data=json.dumps(fields))
    print(response.status_code, response.reason)
    response.raise_for_status()
    data = safe_json(response)
    print(data)
    return data


def save_json(data, filename):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(os.path.join(OUTPUT_DIR, filename), "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [18]:
def main():
    with open(INPUT_CSV, newline='', encoding="utf-8") as csvfile:

        reader = csv.DictReader(csvfile)

        for row in reader:
            user_id = row.get("userId")
            if not user_id:
                continue
            print(f"Processing userId: {user_id}")

            # Step 1: Fetch document
            doc = fetch_document(user_id)
            save_json(doc, f"document_before.json")
            print(f"saved json")

            # Step 2: Update document partially
            update_document(user_id, FIELDS_TO_UPDATE)

            # Step 3: Fetch again after update
            updated_doc = fetch_document(user_id)
            save_json(updated_doc, f"document_after.json")


if __name__ == "__main__":
    main()

Processing userId: 46c59f3e-7c5c-11f0-b37f-b95eaaed324a
{'id': '46c59f3e-7c5c-11f0-b37f-b95eaaed324a', 'dataEntityId': 'b2b_users', 'accountId': '71711ee1-ad02-48dc-a3e5-6fe5a16d55d0', 'accountName': 'helmethouseprod', 'followers': [], 'schemas': [], 'clId': '543d9b52-5056-11f0-b37f-938c2e3386f7', 'costId': '5caf5b97-9d9e-11ed-83ab-0ab848f3a6fb', 'email': 'frankie@factorypowersportslasvegas.com', 'name': 'Frankie M', 'orgId': '597e8c1d-9d9e-11ed-83ab-16c13c50d579', 'roleId': 'customer-approver', 'canImpersonate': None, 'createdBy': '7089111e-3f6a-4e39-8309-f233bb0532a0', 'createdBy_USER': {'Id': '7089111e-3f6a-4e39-8309-f233bb0532a0', 'Login': 'vrn--vtexsphinx--aws-us-east-1--helmethouseprod--master--app_vtex.storefront-permissions@1.45.3', 'Name': None}, 'createdIn': '2025-08-18T17:53:51.0838198Z', 'lastInteractionBy': '6a878a1b-17d5-41e2-ae28-5e3917009a31', 'lastInteractionBy_USER': {'Id': '6a878a1b-17d5-41e2-ae28-5e3917009a31', 'Login': 'vtexappkey-helmethouseprod-WTETKN', 'Name': N